# Expresión génica en hepatocarcinoma (TCGA-LIHC)
### ¿Cambia la expresión de ciertos genes con el grado del tumor?

Cuaderno de la clase **VS Code desde cero** (Viernes de Bioinformática). Es la versión **corta** del análisis (sin funciones), en celdas: ejecutas una celda con **Shift+Enter** y ves su resultado justo debajo.

**Cómo usarlo**
1. Guarda este archivo en la **raíz de tu carpeta de proyecto** (por ejemplo `analisis-hcc/`). El cuaderno crea `data/` y `results/` junto a él: **guárdalo antes de ejecutarlo**.
2. Corre las celdas de la sección 0 una vez para dejar listo el kernel.
3. **Regla del cuaderno: cada línea de código lleva su comentario** (lo que va después de `#`), para que se entienda qué hace sin buscar en otra parte.
4. Sigue de arriba hacia abajo. Si algo sale raro, **Restart** (arriba) y vuelve a correr todo desde el principio.

## 0. Preparar el kernel de Python

Un cuaderno necesita un **kernel**: el Python que ejecuta las celdas. Para correr cualquier celda, VS Code te pide elegir uno (botón **Select Kernel**, arriba a la derecha). Elige cualquier Python que ya tengas; estas celdas dejan listo el kernel de la clase, `Python (viernes-vscode)`, y después cambiamos a él.

**0.1 · ¿Qué Python estoy usando?**

In [ ]:
import sys  # sys: información del Python que está corriendo este cuaderno

print("Python:", sys.version.split()[0])  # solo el número de versión, por ejemplo 3.12.7
print("Ejecutable:", sys.executable)  # ruta exacta del programa python que usa el kernel

**0.2 · ¿Qué bibliotecas faltan?** Buscamos las que necesita la clase sin importarlas todavía.

In [ ]:
from importlib.util import find_spec  # find_spec dice si una biblioteca está instalada, sin importarla

NECESARIAS = ["ipykernel", "pandas", "numpy", "matplotlib"]  # constante: bibliotecas que pide la clase
faltan = [nombre for nombre in NECESARIAS if find_spec(nombre) is None]  # nos quedamos con las que no se encuentran

print("Faltan:", faltan if faltan else "nada, todo está instalado")  # muestra la lista, o un aviso si está vacía

**0.3 · Instalar lo que falte.** El signo `!` al inicio de una línea manda esa orden a la terminal del sistema (solo funciona en cuadernos, no en archivos `.py`). Usamos `sys.executable -m pip` para instalar en **este mismo Python** y no en otro.

In [ ]:
if faltan:  # solo si hay algo que instalar
    # "!" manda la orden a la terminal; {} inserta valores de Python (pip instala en ESTE Python)
    !{sys.executable} -m pip install -q {" ".join(faltan)}
    print("Instalado:", faltan)  # confirma qué se instaló
else:  # si no falta nada
    print("No hay nada que instalar")  # avisa que no se hizo nada

**0.4 · Crear el kernel de la clase.** Registra este Python como un kernel con nombre propio.

In [ ]:
# registra este Python como kernel con nombre propio ("!" = orden de terminal; --user = solo para tu usuario)
!{sys.executable} -m ipykernel install --user --name viernes-vscode --display-name "Python (viernes-vscode)"

**0.5 · Cambiar al kernel nuevo**

1. Arriba a la derecha, clic en el nombre del kernel → **Select Another Kernel…** → **Jupyter Kernel…** → **Python (viernes-vscode)**. Si no aparece en la lista, pulsa el botón de refrescar de esa lista; si sigue sin aparecer, cierra y vuelve a abrir VS Code.
2. Pulsa **Restart** en la barra del cuaderno.
3. Corre la celda de abajo: debe mostrar la ruta del Python del entorno `viernes-vscode` (o el que elegiste, si instalaste en otro).

A partir de aquí, todo el análisis usa ese kernel.

In [ ]:
import sys  # volvemos a importar sys porque el reinicio borró la memoria del kernel

print(sys.executable)  # ruta del Python del kernel nuevo; debe ser la del entorno viernes-vscode

---
## 1. Bibliotecas

Una *biblioteca* es un paquete de funciones que alguien más ya escribió. `import` la trae a tu cuaderno y `as` le pone un apodo corto.

In [ ]:
import os  # os: sirve para crear carpetas
import pandas as pd  # pandas: trabaja con tablas (pd es su apodo)
import matplotlib.pyplot as plt  # matplotlib: dibuja figuras (plt es su apodo)

## 2. Carpetas del proyecto

Cada cosa en su carpeta: los datos en `data/`, las tablas y las figuras que genere el análisis en `results/`. `exist_ok=True` evita el error si la carpeta ya existe. Mira el Explorador: aparecen solas.

In [ ]:
os.makedirs("data", exist_ok=True)  # crea la carpeta de los datos
os.makedirs("results/tablas", exist_ok=True)  # crea la carpeta de las tablas de resultados
os.makedirs("results/figuras", exist_ok=True)  # crea la carpeta de las figuras

## 3. Descargar los datos

Datos reales de hepatocarcinoma (TCGA-LIHC): una fila por paciente, con su grado de tumor y la expresión de 5 genes (ya en escala log2). Pandas puede leer un archivo directo de internet, y luego guardamos una copia en `data/`. Fuente: *The Cancer Genome Atlas Research Network, Cell 2017; 169(7):1327-1341*.

In [ ]:
url = "https://raw.githubusercontent.com/MiguelDiaz02/viernes-vscode/main/data/hcc_tabla.csv"  # dirección del archivo
datos = pd.read_csv(url)  # pandas lee el archivo y lo guarda en la variable datos
datos.to_csv("data/hcc_tabla.csv", index=False)  # guarda una copia en la carpeta data

## 4. Conocer la tabla

`datos` es un *DataFrame*: una tabla con filas y columnas. Antes de analizar, siempre miramos qué tenemos.

In [ ]:
datos.shape  # (filas, columnas): debe dar (371, 9)

In [ ]:
datos.head()  # las primeras 5 filas

In [ ]:
datos["grado"].value_counts()  # cuántos pacientes hay en cada grado (G1 es el más parecido al hígado sano)

## 5. Datos faltantes

En datos reales siempre faltan valores. `isna()` marca los vacíos y `sum()` los cuenta por columna. Sin grado no podemos comparar, así que quitamos esos pacientes con `dropna`.

In [ ]:
datos.isna().sum()  # cuántos valores vacíos hay en cada columna

In [ ]:
datos = datos.dropna(subset=["grado"])  # quita los pacientes que no tienen grado
datos.shape  # ahora debe dar (366, 9)

## 6. Promedio de cada gen por grado

`groupby("grado")` separa a los pacientes por grado y `mean()` promedia cada gen dentro de cada grupo.

In [ ]:
genes = ["AFP", "CA9", "TOP2A", "ALB", "GAPDH"]  # lista con los genes que vamos a comparar
resumen = datos.groupby("grado")[genes].mean()  # promedio de cada gen en cada grado
resumen.round(2)  # muestra la tabla con 2 decimales

In [ ]:
resumen.round(2).to_csv("results/tablas/resumen_por_grado.csv")  # guarda la tabla en results/tablas

**¿Qué vemos?** AFP, CA9 y TOP2A suben en los grados altos; ALB baja un poco; GAPDH (gen de referencia) casi no cambia, como se espera. Son promedios descriptivos: no es una prueba estadística.

## 7. Figura

Una barra por grado y por gen. `plot` dibuja a partir de la tabla; `savefig` guarda la imagen en `results/figuras`.

In [ ]:
resumen.plot(kind="bar", rot=0, figsize=(8, 4))  # barras: una por gen, agrupadas por grado
plt.title("Expresión promedio por grado tumoral (TCGA-LIHC)")  # título de la figura
plt.ylabel("log2(expresión + 1)")  # nombre del eje vertical
plt.legend(title="gen", bbox_to_anchor=(1.02, 1), loc="upper left")  # leyenda fuera de la figura, a la derecha
plt.savefig("results/figuras/expresion_por_grado.png", dpi=150, bbox_inches="tight")  # guarda la imagen
plt.show()  # la muestra aquí en el cuaderno

---
**Lo que quedó en tu carpeta**

```
analisis-hcc/
├── analisis_hcc.ipynb
├── data/        hcc_tabla.csv
└── results/     tablas/resumen_por_grado.csv, figuras/expresion_por_grado.png
```

Siguiente paso (después de la clase): escribir *funciones* para no repetir código. Una versión con funciones está en `respaldo/analisis_hcc_completo.ipynb`.